# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 1: Introduction to Natural Language Processing</font>

# <font color="#003660">Notebook 5: Long-Short Term Memory (LSTM) Networks with Word Embeddings as Features</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... understand the logic behind recurrent neural networks, especially LSTMs. and <br>
        ... are able to train a LSTM with word embeddings as features.
    </font>
</div>
</center>
</p>

# What are Recurrent Neural Networks?

## Simple Recurrent Neural Network (RNN)

A RNN processes sequences by iterating through the sequence elements and maintaining a *state* containing information relative to what it has seen so far. In effect, a RNN layer is a neural network layer with an internal loop, as shown in the figure below.

<br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/rnn1.png"/><br>
<center>Source: Chollet (2021)</center>

The figure below shows a simple RNN unrolled over time. As can be seen from the figure the output of a layer is a combination of

1. its direct data input (`input_t`),
2. the layer's state from the previous timestep (`state_t`), and
3. a bias term (`bo`).

<br><img width=700 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/rnn2.png"/><br>

<center>Source: Chollet (2021)</center>

## Long-Short Term Memory (LSTM) Networks

Compared to a simple RNN layer, a LSTM layer contains one central innovation: A *carry track* that allows to carry over information over time from any previous timestep to the current timestep. Consequently, the output of a layer is a combination of



1. its direct data input (`input_t`),
2. the layer's state from the previous timestep (`state_t`),
3. the input from the carry track (`c_t`), and
4. a bias term (`bo`).

<br><img width=700 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/lstm.png"/><br>
<center>Source: Chollet (2021)</center>

## Comparing RNNs with MLPs

Andrej Karpathy's legendary blog post "The Unreasonable Effectiveness of Recurrent Neural Networks" contains a very informative comparison of RNNs with traditional neural networks: http://karpathy.github.io/2015/05/21/rnn-effectiveness/

# Initialize notebook

Load required packages and initialize the random number generators, so that we get the same results every time we run the notebook.

In [ ]:
%pip install -q torchinfo  # not pre-installed on Colab

In [ ]:
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import mean_absolute_error

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchinfo import summary

In [ ]:
np.random.seed(42)
torch.manual_seed(42)

Use a GPU if one is available: an NVIDIA GPU (`cuda`, e.g., on Colab), Apple silicon (`mps`), or else the CPU.

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Using device:', device)

# Problem description

The dataset contains 100,000 labeled wine reviews (Source: https://www.kaggle.com/datasets/zynicide/wine-reviews) written by professional sommeliers. Besides a text, it contains information about the wine, such as the country of origin, the variety, the winery, the price, and the number of points given by the reviewer. As in Notebook 1b, the goal is to predict the number of points given by the reviewer based on the text of the review. This is a regression problem, so we use the mean absolute error (MAE) as loss.

# Load data

Load data from CSV file.

In [ ]:
corpus = pd.read_csv("https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_01/wine_reviews_train.csv")
corpus.head()

# Prepare data

## Split data

Split data into three sets: training, validation, and test. To speed up training, we only use a random sample of 10,000 reviews for training. We use exactly the same split as in Notebook 1b, so that we can compare the test MAE of the neural network with the one of the linear regression.

In [ ]:
training = corpus.iloc[0:80000,].sample(n=10000, random_state=42) # to speed up training
validation = corpus.iloc[80000:90000,]
test = corpus.iloc[90000:,]

For each dataset, store predictors (the review texts, as a list of strings) and response (the points, as a NumPy array) in separate variables.

In [ ]:
train_texts = training["description"].tolist()
train_points = training["points"].to_numpy(dtype=np.float32)
val_texts = validation["description"].tolist()
val_points = validation["points"].to_numpy(dtype=np.float32)
test_texts = test["description"].tolist()
test_points = test["points"].to_numpy(dtype=np.float32)

Look at the first review of the training set and its points.

In [ ]:
print(train_texts[0])
print('Points:', train_points[0])

## From text to numbers

A neural network can only process numbers, not text. We turn each review into a sequence of integers in three steps:

1. **Tokenization:** split the text into words (*tokens*).
2. **Vocabulary:** keep the 10,000 most frequent words of the training data and replace each word by its index in this vocabulary.
3. **Padding and truncation:** bring all reviews to the same length, so that a minibatch of reviews fits into one tensor.

For steps 1 and 2 we use scikit-learn's `CountVectorizer`, the same tool we used for the bag-of-words models in Notebooks 1a and 1b. There, we used it to count how often each word occurs in a document; here we only need two things it learns along the way: the vocabulary and the tokenizer. As always, we fit it on the **training** texts only.

Unlike in Notebooks 1a and 1b, we don't lemmatize the texts or remove stop words. The embedding layer learns itself which words are similar (e.g., "aroma" and "aromas"), and for a sequence model like the LSTM, small words like "not" and the order of the words can carry important information.

In [ ]:
max_tokens = 10000   # size of the vocabulary

vectorizer = CountVectorizer(max_features=max_tokens, lowercase=True)
vectorizer.fit(train_texts)

`build_analyzer()` returns the tokenizer the vectorizer uses: it lowercases a text and splits it into words. Punctuation is dropped, and so are one-letter words such as "a".

In [ ]:
analyzer = vectorizer.build_analyzer()
print(analyzer(train_texts[0]))

`vocabulary_` is a dictionary that maps each of the 10,000 words to an index between 0 and 9,999 (in alphabetical order). `get_feature_names_out()` lists the words; here we show every 1,000th word.

In [ ]:
vocabulary = vectorizer.vocabulary_
print(len(vocabulary))
print(vectorizer.get_feature_names_out()[::1000])

Now we replace every word of every review by an integer. We reserve two special indices:

- **0** for padding (see below),
- **1** for unknown words, i.e., words that are not in the vocabulary (rare words, or words that only occur in the validation or test data).

The words of the vocabulary therefore get the indices 2 to 10,001 (their vocabulary index + 2). Let's do this first for a single review, word by word: if the word is in the vocabulary, we take its index + 2; otherwise, we take 1. (`int()` turns the NumPy integers stored in `vocabulary_` into plain Python integers.)

In [ ]:
example_words = analyzer(train_texts[0])

example_ids = []
for word in example_words:
    if word in vocabulary:
        example_ids.append(int(vocabulary[word]) + 2)   # known word: its vocabulary index + 2
    else:
        example_ids.append(1)                           # unknown word

print(example_words)
print(example_ids)

To apply the same rule to all reviews, we first store it in a **lookup table**: a Python dictionary `word_to_id` that maps every word of the vocabulary to its integer. `word_to_id.get(word, 1)` looks up a word and returns 1 if the word is not in the table, which is exactly the rule from above.

In [ ]:
word_to_id = {word: int(index) + 2 for word, index in vocabulary.items()}

print(word_to_id['wine'])
print(word_to_id.get('qwertz', 1))   # a word that is not in the vocabulary

Now we go through all reviews of the training, validation and test set and translate each review into a list of integers.

In [ ]:
train_ids = []
for text in train_texts:
    train_ids.append([word_to_id.get(word, 1) for word in analyzer(text)])

val_ids = []
for text in val_texts:
    val_ids.append([word_to_id.get(word, 1) for word in analyzer(text)])

test_ids = []
for text in test_texts:
    test_ids.append([word_to_id.get(word, 1) for word in analyzer(text)])

print(train_ids[0])

The reviews have different lengths.

In [ ]:
train_lengths = [len(ids) for ids in train_ids]
print('Average number of words:', np.mean(train_lengths))
print('Maximum number of words:', np.max(train_lengths))
print('Share of reviews with more than 100 words:', np.mean(np.array(train_lengths) > 100))

We bring all reviews to the same length `max_length = 100`:

- Longer reviews are **truncated**: we keep the *last* 100 words (`ids[-max_length:]`). Almost no wine review is that long.
- Shorter reviews are **padded with zeros at the front**. The real words then always come last, so the last position of every sequence is a real word. This matters for the RNN, which uses its hidden state after the last position to make a prediction (see below).

In [ ]:
max_length = 100

train_truncated = [ids[-max_length:] for ids in train_ids]                                   # keep the last max_length words
train_sequences = np.array([[0] * (max_length - len(ids)) + ids for ids in train_truncated])  # pad with zeros at the front
val_truncated = [ids[-max_length:] for ids in val_ids]
val_sequences = np.array([[0] * (max_length - len(ids)) + ids for ids in val_truncated])
test_truncated = [ids[-max_length:] for ids in test_ids]
test_sequences = np.array([[0] * (max_length - len(ids)) + ids for ids in test_truncated])

print(train_sequences.shape, val_sequences.shape, test_sequences.shape)

Let's look at our example review before and after the transformation: the original text, the padded sequence of 100 integers, and the words behind the integers (we invert the vocabulary to look them up).

In [ ]:
index_to_word = {index + 2: word for word, index in vocabulary.items()}
index_to_word[0] = '<pad>'
index_to_word[1] = '<unknown>'

print(train_texts[0])
print(train_sequences[0])
print([index_to_word[index] for index in train_sequences[0]])

## Data loaders

We now turn the data into PyTorch tensors. The word indices become integers (`torch.long`), because the embedding layer looks up rows by index; the points get an extra dimension (`unsqueeze(1)`), because the network outputs one column.

In [ ]:
X_train_tensor = torch.tensor(train_sequences, dtype=torch.long)
X_val_tensor = torch.tensor(val_sequences, dtype=torch.long)
X_test_tensor = torch.tensor(test_sequences, dtype=torch.long)
y_train_tensor = torch.tensor(train_points).unsqueeze(1)
y_val_tensor = torch.tensor(val_points).unsqueeze(1)
y_test_tensor = torch.tensor(test_points).unsqueeze(1)

The data loaders serve minibatches of 128 reviews, so stochastic gradient descent uses minibatches of 128. The training data is reshuffled in every epoch (with a seeded `generator`, so that the results are reproducible); the validation and test data are only predicted, so their order does not matter. We also predict them in minibatches, because all 10,000 reviews at once would need a lot of memory.

In [ ]:
batch_size = 128

train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size,
                          shuffle=True, generator=torch.Generator().manual_seed(42))
val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size, shuffle=False)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=batch_size, shuffle=False)

print('Minibatches per epoch:', len(train_loader))

# Train model

## Network architecture

We specify the network as a class that inherits from `nn.Module`:

- `nn.Embedding` is a trainable lookup table that maps each word index to a vector of length `embedding_dim`. A review of 100 indices becomes a sequence of 100 word vectors, i.e., a tensor of shape (`max_length`, `embedding_dim`). `padding_idx=0` keeps the vector of the padding index at zero.
- `nn.LSTM` reads this sequence word by word and keeps a hidden state with 32 hidden units, so word order and negation can matter, at the price of much slower training. `batch_first=True` tells it that the first dimension of the input is the minibatch. The LSTM returns its hidden state after every word, with shape (batch, `max_length`, 32).
- We pass only the hidden state after the **last** word, to the output layer (`x[:, -1, :]`): after reading the whole review, it summarizes everything the LSTM has seen. Because we padded at the front, the last position is always a real word of the review, and the LSTM has read the whole review when it gets there. (With padding at the end, the last positions would be zeros for most reviews.)
- A linear output unit for the predicted points.

In [ ]:
class WineLSTM(nn.Module):
    def __init__(self, num_embeddings, embedding_dim, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(embedding_dim, hidden_size, batch_first=True)
        self.output = nn.Linear(hidden_size, 1)

    def forward(self, x):
        x = self.embedding(x)   # (batch, max_length, embedding_dim): one vector per word
        x, _ = self.lstm(x)     # (batch, max_length, hidden_size): hidden state after every word
        x = x[:, -1, :]         # (batch, hidden_size): hidden state after the last word
        x = self.output(x)
        return x

Let's create an instance of the network. The embedding layer needs one row for each of the 10,000 words plus the two special indices (padding and unknown).

We initialize the bias of the output unit with the average points in the training data (about 88). Otherwise, the network would start by predicting values around 0 and would spend many epochs just moving its predictions up to the right range.

We also change the starting values of the word vectors. By default, PyTorch draws them from a standard normal distribution, i.e., values of about ±1 in each of the 300 dimensions. Training works much better here if all word vectors start close to zero, with small random values between −0.05 and 0.05 (the vector of the padding index 0 stays zero).

With `embedding_dim = 300` (the size of popular pretrained embeddings such as GloVe, see Notebook 2), our embedding layer alone has about $10{,}000 \cdot 300 = 3$ million parameters.

In [ ]:
embedding_dim = 300
num_embeddings = max_tokens + 2   # 10,000 words + padding + unknown
mean_points = train_points.mean()

model = WineLSTM(num_embeddings, embedding_dim, hidden_size=32)
model.embedding.weight.data.uniform_(-0.05, 0.05)  # small random starting values for the word vectors
model.embedding.weight.data[0] = 0                 # the padding vector stays zero
model.output.bias.data.fill_(float(mean_points))   # start the output unit at the average rating
model = model.to(device)

`torchinfo.summary` shows the output shape and number of parameters of each layer for a minibatch of 128 reviews. The input consists of word indices, so we pass `dtypes=[torch.long]`.

In [ ]:
summary(model, input_size=(batch_size, max_length), dtypes=[torch.long], device=device)

## Training loop

We minimize the mean absolute error (`nn.L1Loss`). The optimizer is Adam, a variant of stochastic gradient descent, with learning rate ρ = 0.001 (`lr`).

In [ ]:
criterion = nn.L1Loss()   # mean absolute error
optimizer = optim.Adam(model.parameters(), lr=0.001)

Now we write the training loop. This is what Keras' `model.fit()` or the Hugging Face `Trainer` (Session 2) do behind the scenes: the outer loop runs over epochs, the inner loop over minibatches (reset gradients, forward pass, loss, backward pass, update weights), and after each epoch we compute the validation MAE in evaluation mode without gradients.

We also add **early stopping**, written out at the end of each epoch:

- If the validation MAE is the best so far, we store a copy of the current weights. `model.state_dict()` returns the weights themselves, which keep changing during training, so we need `copy.deepcopy` to freeze a snapshot.
- If the validation MAE has not improved for `patience = 2` epochs in a row, we stop training (`break`). We train for at most 10 epochs.
- After training, we restore the best weights with `model.load_state_dict()`.

We use the validation set for early stopping, never the test set.

The LSTM processes the 100 words of a review one after the other, so an epoch takes much longer than for the MLP.

In [ ]:
epochs = 10
patience = 2
train_maes = []   # average training MAE per epoch
val_maes = []     # validation MAE per epoch
best_val_mae = float('inf')
epochs_without_improvement = 0

for epoch in range(epochs):                                   # outer loop: epochs
    # Training
    model.train()
    epoch_loss = 0
    for batch_X, batch_y in train_loader:                     # inner loop: minibatches of 128 reviews
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()                                 # reset gradients
        predictions = model(batch_X)                          # forward pass
        loss = criterion(predictions, batch_y)                # MAE on this minibatch
        loss.backward()                                       # backward pass
        optimizer.step()                                      # update weights
        epoch_loss += loss.item()
    train_maes.append(epoch_loss / len(train_loader))

    # Validation
    model.eval()
    val_abs_error = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            val_abs_error += (model(batch_X) - batch_y).abs().sum().item()   # sum of absolute errors
    val_maes.append(val_abs_error / len(val_points))
    print(f'Epoch {epoch + 1}/{epochs}, Training MAE: {train_maes[-1]:.4f}, Validation MAE: {val_maes[-1]:.4f}')

    # Early stopping
    if val_maes[-1] < best_val_mae:                           # new best model: keep a copy of its weights
        best_val_mae = val_maes[-1]
        best_weights = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement == patience:
            print('Early stopping: no improvement for', patience, 'epochs')
            break

model.load_state_dict(best_weights)                           # restore the weights of the best epoch
print('Restored the weights of epoch', val_maes.index(best_val_mae) + 1, 'with validation MAE', round(best_val_mae, 4))

Plot the learning process.

In [ ]:
plt.plot(range(1, len(train_maes) + 1), train_maes, label='Training')
plt.plot(range(1, len(val_maes) + 1), val_maes, label='Validation')
plt.title('Model MAE')
plt.xlabel('Epoch')
plt.ylabel('Mean absolute error')
plt.legend()
plt.show()

A validation MAE that rises while the training MAE keeps falling indicates overfitting; this is where early stopping kicks in. (The training MAE is averaged over the minibatches of an epoch while the weights are still changing, so it lags a bit behind.)

# Make predictions

The model now has the weights of the best epoch. Make predictions on the test set, minibatch by minibatch, in evaluation mode and without gradients. `.cpu()` moves the predictions back from the GPU, and `torch.cat` puts the minibatches together.

In [ ]:
model.eval()
test_preds = []
with torch.no_grad():
    for batch_X, batch_y in test_loader:
        test_preds.append(model(batch_X.to(device)).cpu())
test_preds = torch.cat(test_preds).numpy().ravel()

Calculate the prediction error (MAE) on the test set.

In [ ]:
print('Test MAE:', mean_absolute_error(test_points, test_preds))

Is this good? As a naive baseline, we always predict the mean points of the training set, no matter what the review says.

In [ ]:
baseline_preds = np.full(len(test_points), train_points.mean())
print('Baseline MAE:', mean_absolute_error(test_points, baseline_preds))

Let's look at a few reviews of the test set with their true and predicted points.

In [ ]:
pd.DataFrame({'review': test_texts[:5],
              'true points': test_points[:5],
              'predicted points': test_preds[:5].astype(float).round(1)})

How does the LSTM compare with the other models? On the same test set (your numbers may differ slightly, e.g., on a different GPU):

| Model | Test MAE |
|---|---|
| Naive baseline (mean points) | ≈ 2.51 |
| Linear regression on word counts (Notebook 1b) | ≈ 1.58 |
| MLP with learned word embeddings (Notebook 4) | ≈ 1.56 |
| LSTM with learned word embeddings (this notebook) | ≈ 1.50 |

The LSTM is the best of our models: reading the words in order pays off a little. But the gain is modest, and the LSTM takes longer to train than the MLP. In Session 2, we will fine-tune a pre-trained transformer on the same task, which has already learned a lot about language from huge amounts of text before it sees a single wine review.

# Exercises

Now it's your turn to improve the model! Here are some ideas, roughly ordered from easy to advanced.

**Change the architecture**

1. **Size of the memory:** Vary the number of hidden units of the LSTM, e.g., `hidden_size=8`, `64`, or `128`. How does the training time change?
2. **Stacked LSTMs:** Stack two LSTM layers with `nn.LSTM(embedding_dim, hidden_size, num_layers=2, batch_first=True)`. The second layer reads the sequence of hidden states produced by the first one.
3. **Bidirectional LSTM:** With `bidirectional=True`, the network reads each review from left to right *and* from right to left. The output then has `2 * hidden_size` columns, so the output layer needs `nn.Linear(2 * hidden_size, 1)`. (Careful: because of the padding at the front, the backward direction's state at the last position has only seen the last word – using `x.mean(dim=1)` instead of `x[:, -1, :]` is a simple way out.)
4. **Other recurrent layers:** Replace `nn.LSTM` with `nn.GRU` (a simplified LSTM with fewer parameters) or `nn.RNN` (a simple RNN without the carry track). How much does the carry track of the LSTM help?
5. **Regularization:** Add a dropout layer (`nn.Dropout(0.3)`) between the LSTM and the output layer. For stacked LSTMs, `nn.LSTM(..., num_layers=2, dropout=0.3)` also applies dropout between the layers.
6. **Additional hidden layer:** Add a hidden layer with ReLU activation (e.g., `nn.Linear(hidden_size, 16)`) between the LSTM and the output layer.
7. **Convolutions instead of recurrence:** Replace the LSTM with a 1D convolution that detects patterns of 5 consecutive words: `nn.Conv1d(embedding_dim, 64, kernel_size=5)`, followed by a max over all positions. Note that `nn.Conv1d` expects the shape (batch, channels, length), so you need `x.transpose(1, 2)` after the embedding layer. How does it compare in accuracy and speed?

**Change the data and training**

8. **Sequence length:** An LSTM processes a review word by word, so `max_length` directly affects training time. Try `max_length = 50` and `200`.
9. **More data:** We train on a sample of only 10,000 reviews. Use 40,000 or all 80,000 training reviews. Does the LSTM benefit more from additional data than the MLP from Notebook 4?
10. **Optimization:** Change the learning rate (`lr=0.0001` or `0.01`) or `patience`.

**Reflect**

11. Compare your best LSTM with your best MLP from Notebook 4 and the linear regression from Notebook 1b: how much better is it, and how much longer does it take to train? When would the extra effort be worth it?

**How to work on the exercises:** Copy the class definition and the cell that creates the model into the code cell below, give the class a new name (e.g., `WineLSTM2`), and change the architecture. Then create a new optimizer (`optimizer = optim.Adam(model.parameters(), lr=0.001)`), rerun the training loop, and calculate the MAE on the test set. Change only *one thing at a time*, and keep track of your results (test MAE, number of parameters from `summary()`, training time) in a small table – that's the only way to learn which change actually helped.

In [ ]:
# YOUR CODE GOES HERE